# D2VLM on Kaggle — environment, model & asset setup

This notebook follows the official D2VLM README (https://github.com/nusnlp/d2vlm).
It only performs the steps that are impractical to prepare by hand:

1. Build the strict **Python 3.12 environment** (torch 2.4.0 / transformers 4.44.2 / nncore 0.4.5 / decord) as requested by the README.
2. Download the **D2VLM model weights** (`wenzhengzeng/D2VLM-Models`, ~10 GB) — too large to upload.
3. Download the annotation-only assets needed for **Charades-STA**:
   - `ShuhuaiRen/TimeIT` Charades-STA caption annotations
4. Verify the environment and provide ready-to-run inference/eval cells.

### Kaggle prerequisites (set in the notebook sidebar)
- **Accelerator:** GPU **T4 x2**
- **Internet:** **On** (needed for the downloads below)
- Attach your uploaded Kaggle dataset for the post-processed Charades-STA root (`charades/`).

> The config cell below sets environment variables that the later `%%bash` cells inherit, so run the cells **in order**.

In [1]:
%%bash
# Update package lists and install Python 3.10
apt-get update -y
apt-get install -y python3.12 python3.12-venv python3.12-dev

# Download and install pip for Python 3.10
wget -q https://bootstrap.pypa.io/get-pip.py
python3.12 get-pip.py --quiet
rm get-pip.py

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1,912 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,053 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:13 http://archive.ubuntu.com/ubuntu noble-

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manag

In [2]:
%%bash
# Create virtual environment using python3.12
python3.12 -m venv /kaggle/working/py312env

# Upgrade pip inside the python3.12 environment
/kaggle/working/py312env/bin/pip install --upgrade pip



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 8.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0


In [3]:
!nvidia-smi
!df -h /kaggle/working /kaggle/temp
!python --version


Sun Oct 11 12:33:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Build the Python 3.12 conda environment

In [4]:
%%bash
# set -e
# source /opt/conda/etc/profile.d/conda.sh
# conda activate d2vlm
# PyTorch 2.4.0 (CUDA 12.1 wheels, matching the README)
/kaggle/working/py312env/bin/pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 291.1 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 304.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 136.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 647.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 180.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 581.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 189.4 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 187.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 211.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 179.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 152.0 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB

In [5]:
%%bash
set -e
# source /opt/conda/etc/profile.d/conda.sh
# conda activate d2vlm

# Pinned stack from requirements.txt (deepspeed omitted: training-only, not needed for inference)
/kaggle/working/py312env/bin/pip install --no-cache-dir \
  numpy==1.26.4 \
  transformers==4.44.2 \
  accelerate==0.33.0 \
  peft==0.12.0 \
  sentence-transformers==3.0.1 \
  sentencepiece==0.2.0 \
  nncore==0.4.5 \
  tabulate psutil tqdm debugpy huggingface_hub hf_transfer

/kaggle/working/py312env/bin/pip install --no-cache-dir 'git+https://github.com/salaniz/pycocoevalcap.git@a24f74c'

# decord==0.6.0 has no cp312 wheel; fall back to eva-decord (drop-in decord API)
/kaggle/working/py312env/bin/pip install --no-cache-dir decord==0.6.0 || pip install --no-cache-dir eva-decord

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 9.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 13.9 MB/s  0:00:00
   

  Running command git clone --filter=blob:none --quiet https://github.com/salaniz/pycocoevalcap.git /tmp/pip-req-build-nxo7bfhj
  Running command git checkout -q a24f74c


In [6]:
%%bash
# apt-get update -y
apt-get install -y openjdk-17-jdk-headless
java -version

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 146 not upgraded.
Need to get 120 MB of archives.
After this operation, 272 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jre-headless amd64 17.0.20.1+1-1~24.04 [48.1 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jdk-headless amd64 17.0.20.1+1-1~24.04 [71.6 MB]
Fetched 120 MB in 3s (37.9 MB/s)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 129826 files and dire

openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)


In [7]:
import os

# Set Java home directory
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# Verify Python can detect Java 17
import subprocess
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-24.04-Ubuntu, mixed mode, sharing)




## 2. Configure paths

Edit the `/kaggle/input/...` paths to match the Kaggle datasets you attach. The variables set here are inherited by every later `%%bash` cell.

In [8]:
import os

# ---- working directories (persisted under /kaggle/working) ----
os.environ['KAGGLE_WORK'] = '/kaggle/working'
os.environ['D2VLM_ROOT'] = '/kaggle/working/d2vlm'          # repo clone
os.environ['MODELS_DIR'] = '/kaggle/working/models'         # downloaded HF weights
os.environ['ANNOS_DIR']  = '/kaggle/working/annotations'    # downloaded HF annotations
os.environ['HF_HOME']    = '/kaggle/working/hf_cache'       # persisted HF cache
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Put the d2vlm conda env first on PATH so !cmd / %%bash use Python 3.12
env_bin = '/opt/conda/envs/d2vlm/bin'
os.environ['PATH'] = env_bin + os.pathsep + os.environ.get('PATH', '')
os.environ['CONDA_PREFIX'] = '/opt/conda/envs/d2vlm'

# ==== ATTACH THE CHARADES-STA KAGGLE DATASET (edit the /kaggle/input/... path) ====
os.environ['OTHER_BENCH_ROOT'] = '/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize'  # contains charades/
os.environ['PRED_ROOT'] = '/kaggle/working/preds'

# ---- model selection: SFT `d2vlm` or `d2vlm_mcqa_enhanced` ----
os.environ['MODEL_SUBDIR'] = 'd2vlm'        # 'd2vlm_mcqa_enhanced' for the MCQA-enhanced model
os.environ['ARCH_MODULE'] = 'd2vlm_arch'    # 'd2vlm_fpo_arch' for the MCQA-enhanced model

# ---- retrieval hyperparameters (tunable) ----
os.environ['NUM_CANDIDATES'] = '5'          # number of sampled candidates per query (Recall@NUM_CANDIDATES)
os.environ['TEMPERATURE'] = '0.7'           # sampling temperature for the candidates

# ---- annotation paths (downloaded below; set here so later cells can use them) ----
os.environ['TIMEIT_DIR'] = os.path.join(os.environ['ANNOS_DIR'], 'TimeIT')
os.environ['CHARADES_GT'] = os.path.join(os.environ['TIMEIT_DIR'], 'data', 'temporal_video_grounding', 'charades', 'charades_annotation', 'test.caption_coco_format.json')

for d in [os.environ['MODELS_DIR'], os.environ['ANNOS_DIR'], os.environ['HF_HOME'], os.environ['PRED_ROOT']]:
    os.makedirs(d, exist_ok=True)

for k in ['D2VLM_ROOT', 'MODELS_DIR', 'ANNOS_DIR', 'HF_HOME',
          'OTHER_BENCH_ROOT', 'PRED_ROOT', 'MODEL_SUBDIR', 'ARCH_MODULE',
          'NUM_CANDIDATES', 'TEMPERATURE']:
    print(k, '=', os.environ[k])

D2VLM_ROOT = /kaggle/working/d2vlm
MODELS_DIR = /kaggle/working/models
ANNOS_DIR = /kaggle/working/annotations
HF_HOME = /kaggle/working/hf_cache
ETBENCH_VIDEOS = /kaggle/input/etbench-videos-compressed/videos_compressed
OTHER_BENCH_ROOT = /kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize
PRED_ROOT = /kaggle/working/preds
MODEL_SUBDIR = d2vlm
ARCH_MODULE = d2vlm_arch


## 3. Clone the D2VLM repository

In [9]:
%%bash
set -e
if [ ! -d $D2VLM_ROOT/.git ]; then
  git clone https://github.com/nusnlp/d2vlm.git $D2VLM_ROOT
else
  echo 'repo already present'
fi
cd $D2VLM_ROOT && git log --oneline -1

756d340 Add License file


Cloning into '/kaggle/working/d2vlm'...


## 4. Download the model weights (cannot be uploaded by hand)

`wenzhengzeng/D2VLM-Models` -> only the `d2vlm*/` folder selected in the config cell. The repo also ships TF events/runs/etbench folders that are not needed for inference and are skipped here.

In [10]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
import os
from huggingface_hub import snapshot_download

sub = os.environ['MODEL_SUBDIR']
snapshot_download(
    repo_id='wenzhengzeng/D2VLM-Models',
    local_dir=os.environ['MODELS_DIR'],
    allow_patterns=[
        f'{sub}/config.json',
        f'{sub}/generation_config.json',
        f'{sub}/model-*.safetensors',
        f'{sub}/model.safetensors.index.json',
        f'{sub}/tokenizer.json',
        f'{sub}/tokenizer.model',
        f'{sub}/tokenizer_config.json',
        f'{sub}/special_tokens_map.json',
        f'{sub}/added_tokens.json',
    ],
    max_workers=8,
)
print('model ready at', os.path.join(os.environ['MODELS_DIR'], sub))
PY

model ready at /kaggle/working/models/d2vlm


## 5. Download the annotation assets (annotations are small; videos are attached separately)

In [11]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
import os
from huggingface_hub import snapshot_download

# TimeIT annotations for Charades-STA
snapshot_download(
    repo_id='ShuhuaiRen/TimeIT',
    repo_type='dataset',
    local_dir=os.environ['TIMEIT_DIR'],
    allow_patterns=[
        'data/temporal_video_grounding/charades/charades_annotation/test.caption_coco_format.json',
    ],
)
print('TimeIT Charades-STA annotations ready')
PY

D2VLM-Dataset annotations ready
ETBench annotations ready
TimeIT annotations ready


## 6. Verify the environment

In [13]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
import importlib.metadata as md
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())
import transformers
print('transformers', transformers.__version__)
for p in ['decord', 'eva-decord', 'nncore', 'peft', 'sentence-transformers', 'pycocoevalcap']:
    try:
        print(p, md.version(p))
    except Exception:
        pass
import decord, nncore  # noqa: F401
print('imports OK')
PY

torch 2.4.0+cu121 | cuda True | gpus 2
transformers 4.44.2
decord 0.6.0
nncore 0.4.5
peft 0.12.0
sentence-transformers 3.0.1
pycocoevalcap 1.2
imports OK


In [14]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 --version

Python 3.12.3


## 7. Run Charades-STA inference & evaluation

These cells call the exact entry points from the README with Kaggle paths and 2 T4 chunks.

> The repo's `.sh` wrappers contain hard-coded `/storage/...` paths, so the cells below invoke `d2vlm/eval/infer_etbench_decouple.py` directly. No repository file is modified.

Sampling is controlled by the `TEMPERATURE` and `NUM_CANDIDATES` hyperparameters in the config cell: the model draws `NUM_CANDIDATES` samples per query, and the Charades-STA metric reports **Recall@NUM_CANDIDATES** (any of the sampled intervals matching) alongside the standard IoU@0.3/0.5/0.7 (computed on the first candidate).

### 7a. Charades-STA

Requires the post-processed root (`charades/compressed_videos`, `charades/inference_input.json`) produced by `other_benchmark_organize/charades/run.sh`, attached at `$OTHER_BENCH_ROOT`.

In [18]:
%%bash
set -e
cd $D2VLM_ROOT
export PYTHONPATH="./:$PYTHONPATH"
PRED_DIR="$PRED_ROOT/charades"
mkdir -p "$PRED_DIR"
for IDX in 0 1; do
  CUDA_VISIBLE_DEVICES=$IDX MASTER_PORT=$((29502+IDX)) /kaggle/working/py312env/bin/python3.12 d2vlm/eval/infer_etbench_decouple.py \
    --anno_path "/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/charades/inference_input.json" \
    --data_path "$OTHER_BENCH_ROOT" \
    --pred_path "$PRED_DIR" \
    --model_path "$MODELS_DIR/$MODEL_SUBDIR" \
    --arch_module "$ARCH_MODULE" \
    --num_candidates "$NUM_CANDIDATES" \
    --temperature "$TEMPERATURE" \
    --chunk 2 --index $IDX &
done
wait
/kaggle/working/py312env/bin/python3.12 all_benchmark_eval/charades/convert_result_in_timechat_format.py --pred_path "$PRED_DIR"
/kaggle/working/py312env/bin/python3.12 all_benchmark_eval/charades/eval_in_timechat_format.py \
  --pred_file "$PRED_DIR/result_converted.json" \
  --gt_file "$CHARADES_GT"

Chunk: 2 Index: 0 Output Path: /kaggle/working/preds/charades/etbench_0.json
Loading full model from /kaggle/working/models/d2vlm...
Chunk: 2 Index: 1 Output Path: /kaggle/working/preds/charades/etbench_1.json
Loading full model from /kaggle/working/models/d2vlm...
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 1860/1860, 0.5 task/s, elapsed: 59m55s, eta: 0s
[>>>>>>>>>>>>>>>>>>>>>>>>>>>>>] 1860/1860, 0.5 task/s, elapsed: 1h34s, eta: 0s
Total number of files: 2
Loading /kaggle/working/preds/charades/etbench_1.json...
Loading /kaggle/working/preds/charades/etbench_0.json...
Total number of samples: 3720
Start evaluation...
tvg_charades_sta: 3720
# pred video timestamps 3720; # gt video timestamps 3720
3720 10733.702083544576 {0: 3720, 0.3: 2491, 0.5: 1891, 0.7: 962, 'miou': 1669.7020835445767}
IOU 0.3: 66.96236559139786
IOU 0.5: 50.833333333333336
IOU 0.7: 25.86021505376344
 mIOU: 0.4488446461141335


/kaggle/working/py312env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/kaggle/working/py312env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 3/3 [00:27<00:00,  9.01s/it]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequenc

/kaggle/input/datasets/nguynhongthunpht/d2vlm-charades-youcook2/other_benchmark_organize/charades/inference_input.json

/kaggle/input/datasets/nguynhongthunpht/d2vlm-charades-youcook2/other_benchmark_organize/charades/compressed_videos/0FJSU.mp4

## Notes / known issues

- **Videos are not downloaded here.** Attach the post-processed Charades-STA root as a Kaggle Dataset and point `OTHER_BENCH_ROOT` at it.
- **decord**: `decord==0.6.0` has no Python 3.12 wheel; the install falls back to `eva-decord` (same `decord` module API).
- **nncore 0.4.5** compiles a small CUDA op (`temporal_iou`) against the installed torch during `pip install`; that is expected and may take a few minutes.
- **deepspeed** is intentionally not installed (training-only; not used for inference).
- **Retrieval inference**: each query is sampled `NUM_CANDIDATES` times at `TEMPERATURE`, so inference costs ~`NUM_CANDIDATES`× the single-sample run. Lower `NUM_CANDIDATES` or `TEMPERATURE` for a faster/less diverse run.
- **Recall@K**: the Charades-STA metric keeps the original IoU@0.3/0.5/0.7 (first candidate) and adds Recall@K (@0.3/0.5/0.7, plus mIoU@K) using any of the `NUM_CANDIDATES` sampled intervals.
- **Persistence**: commit the notebook / save `Version` to keep `/kaggle/working` (model, preds, cache) between sessions.